# TP : projections climatiques
## Estimation des incertitudes climatiques dans le cas des projections de la température globale

Enora Cariou (enora.cariou@meteo.fr)

Votre notebook, renommé suivant le format `notebook_VOTRE_NOM_DE_FAMILLE.ipynb` est à déposer sur l'espace suivant : https://drive.google.com/drive/folders/1aHuT1AImUva7QmAFISaYpErH-HNZUXWW?usp=drive_link

### Rappels du cours sur les projections futures et incertitudes

**Rappel de terminologie :**

- Climat = état moyen + variabilité climatique
- Variabilité climatique = variabilité interne + forçages externes
- Forçages externes = forçages naturels + forçages anthropiques
- Variabilité interne + forçages naturels = variabilité naturelle

La précision des projections climatiques fournies par les modèles de climat dépend de **l'incertitude climatique** qui est fonction de trois facteurs :
1. le scénario d'émissions des gaz à effet de serre (GES) et des aérosols anthropiques, c'est-à-dire des *forçaces externes* d'origine anthropique, dont la concentration va être modifié au cours des prochaines décennies : **<span style="color: green;">l'incertitude scénario</span>**
2. **<span style="color: orange;">la variabilité interne</span>** du système climatique
3. l'erreur de modélisation de la variabilité climatique : **<span style="color: blue;">l'incertitude modèle</span>**

L'incertitude scénario due aux forçages anthropiques qui modifient l'équilibre radiatif du système climatique est modulée par le choix d'une trajectoire socio-économique plus ou moins émettrice de GES.

L'incertitude due à la variabilité interne englobe les fluctuations climatiques qui se produisent naturellement en l'absence de toute modification des forçages radiatifs.

L'incertitude du modèle quantifie l'erreur de modélisation dans la simulation des réponses climatiques aux forçages naturels (activité solaire, aérosols naturels etc.) et anthropiques.

#### L'amplitude des incertitudes dépend de l'échelle spatio-temporelle

Pour un scénario d'émissions donné, nous savons qu'à l'échelle locale, l'incertitude liée à la variabilité interne est généralement plus grande que l'incertitude modèle à une échéance décennale. A mesure que l'échelle spatio-temporelle s'agrandit, la variabilité interne locale est moyennée et est réduite laissant apparaître la réponse du système aux forçages externes anthropiques.

Ainsi, plus l'échelle spatio-temporelle devient grande, plus l'incertitude due à la variabilité interne diminue et plus l'incertitude modèle augmente, si bien que dans le cas des projections de la température globale, la seconde devient prédominante devant la première :

![alt text](data/frac_var.png "Title")

### Objectifs du TP :
- Estimer les incertitudes associées aux projections de températures, en estimant les contributions :
    - de la variabilité interne
    - de la trajectoire socio-économique future (scénario)
    - des biais des modèles
- Quantifier la sensibilité de ces incertitudes à l'échelle spatio-temporelle

### Données à disposition :
Température moyenne à 2 m de 1850 à 2100 pour 20 modèles de climat, et
3 scénarios socio-économiques (Shared Socio-Economic Pathway): SSP1-2.6, SSP2-4.5, SSP5-8.5.
Toutes les données correspondantes sont préparées dans les fichiers sous le format ``T_*ssp*.asc``.
On étudie le champ de température moyennée sur le globe, et sur la France en été (moyenne JJA, Juin-Juillet-Août) et en hiver (moyenne DJF, Décembre-Janvier-Février).

### Chargement/installation des librairies python

La cellule suivante contient les librairies à charger, elle n'est pas à modifier.

***Si l'execution de la cellule ne fonctionne pas:***
- Installer les packages manquant en tapant dans un terminal, sans être connecté au VPN Météo-France :

`pip install --user le_package_en_question`

- Redémarrer le noyau dans le menu ci-dessus.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

### Chargement des variables, et prise en main de xarray

Dans un premier temps, il s'agit de charger les champs de température pour les 3 scénarios considérés :

In [ ]:
# Set the path where data are located
path_data = "./data/"

# load ascii files

# Global temperature
T_glo_ssp26 = pd.read_csv(path_data+"T_Glo_ANN_ssp126.asc", sep=" ")-273.15
T_glo_ssp45 = pd.read_csv(path_data+"T_Glo_ANN_ssp245.asc", sep=" ")-273.15
T_glo_ssp85 = pd.read_csv(path_data+"T_Glo_ANN_ssp585.asc", sep=" ")-273.15

# Temp over France in JJA
T_fr_jja_ssp26 = pd.read_csv(path_data+"T_fr_JJA_ssp126.asc", sep=" ")-273.15
T_fr_jja_ssp45 = pd.read_csv(path_data+"T_fr_JJA_ssp245.asc", sep=" ")-273.15
T_fr_jja_ssp85 = pd.read_csv(path_data+"T_fr_JJA_ssp585.asc", sep=" ")-273.15

# Temp over France in DJF
T_fr_djf_ssp26 = pd.read_csv(path_data+"T_fr_DJF_ssp126.asc", sep=" ")-273.15
T_fr_djf_ssp45 = pd.read_csv(path_data+"T_fr_DJF_ssp245.asc", sep=" ")-273.15
T_fr_djf_ssp85 = pd.read_csv(path_data+"T_fr_DJF_ssp585.asc", sep=" ")-273.15


Puis on vérifie que les variables chargés ont bien les bonnes coordonnées et dimensions.

In [ ]:
# Verify fields dimensions
T_ssp26 = T_fr_jja_ssp26
T_ssp45 = T_fr_jja_ssp45
T_ssp85 = T_fr_jja_ssp85

T_ssp26

In [ ]:
# Get dimension numbers and names
years = T_ssp26.index.values
models = T_ssp26.columns.values
nyr = len(years)
nmod = len(models)

On peut maintenant commencer à estimer l'incertitude dans notre ensemble !

### Séparation de la variabilité interne et de la réponse aux forçages externes :

La variabilité de la température globale est supposée décomposable comme combinaison linéaire d’une variabilité forcée par les forçages externes (gaz à effet de serre, aérosols, cycle solaire) et d’une variabilité naturelle, intrinsèque au système climatique (variabilité interne).

Les projections de température $X$ pour chaque modèle $m$, scénario $s$ et échéance $t$ peuvent alors s’écrire de la manière suivante :

$$ X(t, m, s) = x(t, m, s) + \epsilon(t, m, s) $$

- La réponse aux forçages externes est notée $x$. Elle correspond à un lissage de $X$ sur l’ensemble de la période projetée, de telle sorte à filtrer la variabilité interne interannuelle au premier ordre.
- Le résidu est noté $\epsilon$ et correspond à la variabilité interne.

Voyons d'abord à quoi ressemble la série temporelle brute de la température en France en JJA pour le modèle CNRM-CM6-1 pour les trois scénarios envisagés.

<div style="background:rgb(230,255,230); padding:3px;">    
Tracer la série temporelle brute de la température en France en JJA pour le modèle CNRM-CM6-1 pour les trois scénarios envisagés
</div>

In [ ]:
# Plot
plt.plot(??, label='SSP1-2.6') #A COMPLETER
plt.plot(??, label='SSP2-4.5') #A COMPLETER
plt.plot(??, label='SSP5-8.5') #A COMPLETER
plt.title("Simulated temperature over France in CNRM-CM6-1")
plt.xlabel('Years') 
plt.ylabel('°C')
plt.legend()

Pour chacune de ces trois séries de $X$, la variabilité interne $\epsilon$ se superpose à une réponse de la température aux forçages externes anthropiques $x$, qui se matérialise par une tendance croissante.

#### Estimation de la réponse aux forçages externes

L'étape suivante consiste à isoler $x$ de $\epsilon$. Pour cela, nous estimons $x$ en approximant $X$ par un polynôme d'ordre 2 (= tendance quadratique) pour lisser la série temporelle.

In [ ]:
# Fit quadratic parameters on raw time series
deg = 2
coef_T_ssp26 = np.polyfit(years,T_ssp26,deg)
coef_T_ssp45 = np.polyfit(years,T_ssp45,deg)
coef_T_ssp85 = np.polyfit(years,T_ssp85,deg)

# initialize the forced response
ant_T_ssp26 = T_ssp26.copy()
ant_T_ssp45 = T_ssp45.copy()
ant_T_ssp85 = T_ssp85.copy()

# compute the time series for each model
for imod in range(nmod):
    ant_T_ssp26[models[imod]] = np.polyval(coef_T_ssp26[:,imod], years)
    ant_T_ssp45[models[imod]] = np.polyval(coef_T_ssp45[:,imod], years)
    ant_T_ssp85[models[imod]] = np.polyval(coef_T_ssp85[:,imod], years)

# Plot for CNRM-CM6
plt.plot(ant_T_ssp26["CNRM-CM6-1"], label='SSP1-2.6')
plt.plot(ant_T_ssp45["CNRM-CM6-1"], label='SSP2-4.5')
plt.plot(ant_T_ssp85["CNRM-CM6-1"], label='SSP5-8.5')
plt.title("Estimates of the forced temperature response in CNRM-CM6-1")
plt.xlabel('Years') 
plt.ylabel('°C')
plt.legend()


### Question

Ce lissage de $X$ vous parait-il correct ? Pourquoi ?


<div style="background:rgb(230,255,230); padding:3px;">    
Votre réponse ici
</div>

Choisissez un lissage adapté pour continuer le TP

#### Estimation de la variabilité interne

Maintenant que l'on a isolé la réponse $x$, la variabilité interne $\epsilon$ se déduit facilement en calculant la différence entre $X$ et $x$.

Compléter le code afin de calculer la variabilité interne pour chaque scénario


In [ ]:
# get internal variability 
iv_T_ssp26 = ?? #A COMPLETER
iv_T_ssp45 = ?? #A COMPLETER
iv_T_ssp85 = ?? #A COMPLETER

In [ ]:
# plot for CNRM-CM6 (historical period)
plt.figure(figsize=(12,8))
plt.plot(iv_T_ssp26["CNRM-CM6-1"].loc[1850:2014], label='SSP1-2.6')
plt.plot(iv_T_ssp45["CNRM-CM6-1"].loc[1850:2014], label='SSP2-4.5')
plt.plot(iv_T_ssp85["CNRM-CM6-1"].loc[1850:2014], label='SSP5-8.5')
plt.title("Estimates of temperature internal variability over France in CNRM-CM6-1")
plt.xlabel('Years') 
plt.ylabel('°C')
plt.legend()


### Questions

- Qu'est-ce qui expliquerait les fluctuations de température sur la France entre 1850 et 2014 (période historique) constatées sur ces courbes ?
- Est-il possible de correctement estimer la variabilité interne sur cette période ?
- Si vous aviez un modèle de climat à disposition, que feriez-vous pour obtenir une meilleure estimation de la variabilité interne ?

<div style="background:rgb(230,255,230); padding:3px;">    
Votre réponse ici
</div>

### Estimation de l'incertitude climatique

Nous reprenons la décomposition en sources d'incertitudes formulée au début du TP.

L’incertitude $\color{orange}{V}$ due à la variabilité interne est considérée ici comme constante (indépendante de $t$) et indépendante du scénario d'émission (indépendante de $s$). Elle est définie à partir de la moyenne multi-modèles de la variance de $\epsilon$ :

$$\color{orange}{V = \frac{1}{N_{mod}} \sum_{m=1}^{N_{mod}} \textrm{Var}_{t,s}[\epsilon(m)]}$$

où $N_{mod}$ correspond au nombre de modèles.

In [ ]:
# Concatenate the three scenarios
iv_T = pd.concat([iv_T_ssp26,iv_T_ssp45,iv_T_ssp85])

# Compute the variance across the scenarios and the years
iv_unc_variance = iv_T.var()

# Get the multi-model mean
iv_unc = iv_unc_variance.mean()
iv_unc

L’incertitude modèle $\color{blue}{M}$ repose sur la dispersion des réponses $x$ simulées par les différents modèles disponibles. Elle est définie pour chaque échéance $t$ :

$$\color{blue}{M(t) = \frac{1}{N_{scen}} \sum_{s=1}^{N_{scen}} \textrm{Var}_{m}[x(t,s)]}$$

où $N_{scen}$ désigne le nombre de scénarios.

In [ ]:
# Model uncertainty for each scenario: variance across the models
mod_unc_T_ssp26 = ant_T_ssp26.var(axis=1)
mod_unc_T_ssp45 = ant_T_ssp45.var(axis=1)
mod_unc_T_ssp85 = ant_T_ssp85.var(axis=1)

# Mean across the scenarios
mod_unc = 1/3 * (mod_unc_T_ssp26+mod_unc_T_ssp45+mod_unc_T_ssp85)
mod_unc

L’incertitude scénario $\color{green}{S}$ est quant à elle définie comme la variance entre les scénarios des moyennes multi-modèles de $x$. Elle dépend de $t$ :

$$\color{green}{S(t) = \textrm{Var}_{m}\left[\frac{1}{N_{mod}} \sum_{m=1}^{N_{mod}} x(t,m)\right]}$$

In [ ]:
# Scenario uncertainty for each year: multi-model mean across the forced response
mod_avg_T_ssp26 = ant_T_ssp26.mean(axis=1)
mod_avg_T_ssp45 = ant_T_ssp45.mean(axis=1)
mod_avg_T_ssp85 = ant_T_ssp85.mean(axis=1)

# Concatenate the three scenarios
mod_avg_T = pd.concat([mod_avg_T_ssp26,mod_avg_T_ssp45,mod_avg_T_ssp85],axis=1,keys=['SSP26','SSP45','SSP85'])

# Variance across the scenarios
scen_unc = mod_avg_T.var(axis=1)
scen_unc

Les incertitudes sont supposées comme étant additives et n’interagissent pas entre-elles ; l'incertitude totale est alors définie comme la somme $T$ des incertitudes calculées précédemment :
$$ T(t) = \color{orange}{V} + \color{blue}{M(t)} + \color{green}{S(t)}$$

In [ ]:
# Compute the total variance
total_unc = ?? #A COMPLETER

### Question

- Vérifiez graphiquement que l'incertitude total $T$ croît avec l'échéance.

In [ ]:
# plot over the projected period


Pour identifier l'importance de chaque source d'incertitude, on peut normaliser chacune d'entre-elles par l'incertitude totale, pour obtenir des pourcentages relatifs d'incertitude.

In [ ]:
# Normalize each uncertainty by T
iv_unc_relative = iv_unc/total_unc*100
mod_unc_relative = mod_unc/total_unc*100
scen_unc_relative = scen_unc/total_unc*100

unc_relative = pd.concat([mod_unc_relative,scen_unc_relative,iv_unc_relative],axis=1,
                         keys=['model uncertainty','scenario uncertainty','internal variability'])

unc_relative

Visuellement, on peut finalement refaire la représentation graphique présentée au début du TP.

In [ ]:
ax = unc_relative.loc[2023:2100].plot(kind='area', stacked=True, color=['royalblue','seagreen','darkorange'],
                                      title='Uncertainty in the projected temperature JJA')

ax.set_ylabel('Fraction of total variance (%)')
ax.set_xlabel('Years')
plt.show()

### Question

Enregistrer la figure ci-dessus, et faîtes le même exercice d'estimation de l'incertitude climatique pour :
- la température en France en DJF
- la température annuelle globale (moyenne planétaire)

Quelles sont les différences avec les incertitudes associée à la température en France en JJA ? Comment les expliquez-vous ?
Comparer les courbes d'incertitudes totales T associées à la température en France en JJA et en DJF. Comment expliquez-vous les différences?


<div style="background:rgb(230,255,230); padding:3px;">    
Vos réponses ici
</div>